# SPORTZBOT RAG — Reviewed Results Walkthrough

This companion notebook explains the verified architecture and observable outputs in the repository. The project does **not** contain a formal offline accuracy benchmark, so this walkthrough avoids claiming a measured accuracy improvement that was never evaluated.


## 1. Establish the knowledge-base scale


In [1]:
# These counts come from the repository's indexed knowledge-base build.
print("source_documents: 10")
print("indexed_chunks: 57")
print("embedding_model: all-MiniLM-L6-v2")


source_documents: 10
indexed_chunks: 57
embedding_model: all-MiniLM-L6-v2


### What this tells us

The system is intentionally narrow and curated: ten source documents become 57 searchable chunks. That makes the chatbot easier to audit than an unconstrained web-search assistant, but it also means coverage is limited to what has been written into the repository's sports knowledge base.


## 2. Make the retrieval policy visible


In [1]:
# MMR is configured to return six chunks after considering a broader pool of twelve.
retrieval = {
    "search_type": "MMR",
    "k": 6,
    "fetch_k": 12,
}
for key, value in retrieval.items():
    print(f"{key}: {value}")


search_type: MMR
k: 6
fetch_k: 12


### What this tells us

The retriever does not simply take the six closest embeddings. It first considers twelve candidates and uses Maximum Marginal Relevance to favor a more diverse final set of six. That design is sensible for questions that need context from more than one source, although the repository does not yet include a measured MMR-vs-similarity benchmark.


## 3. Show the generation configuration and guardrail


In [1]:
# Low temperature favors stable answers, while the prompt requires a knowledge-base fallback.
print("llm: llama3.2 via Ollama")
print("temperature: 0.2")
print("fallback: I do not have that in my knowledge base yet.")


llm: llama3.2 via Ollama
temperature: 0.2
fallback: I do not have that in my knowledge base yet.


### What this tells us

The project is designed for grounded, relatively deterministic responses. The explicit fallback is particularly important: when retrieval does not provide evidence, the desired behavior is to admit the gap rather than improvise a sports fact from model memory.


## 4. Interpret the health endpoint correctly


In [1]:
# The API exposes operational status and the current vector-store size.
health = {"status": "online", "model": "llama3.2", "chunks": 57}
print(health)


{'status': 'online', 'model': 'llama3.2', 'chunks': 57}


### What this tells us

A healthy endpoint confirms that the API, local LLM configuration, and vector store are available. It is an **operational check**, not evidence that generated answers are correct. Answer quality would require a separate evaluation set with retrieval relevance, faithfulness, and answer-quality measures.


## 5. Current evidence boundary

The repository demonstrates a complete local RAG application—document loading, chunking, embeddings, ChromaDB, MMR retrieval, prompt grounding, Llama 3.2 generation, FastAPI, and a custom UI.

What it does **not** yet demonstrate is a quantified hallucination rate or a controlled comparison showing how much RAG improves answer accuracy. That would be the next rigorous evaluation step.
